# commgwas tutorial: SLE MAGMA signal in blood communication

This notebook uses the independent package under `CommGWAS/package/`. It demonstrates the tissue, cell-type-pair, LR-pair, and NicheNet downstream levels. Every function returns a pandas DataFrame and does not create an FDR column.

In [ ]:
from pathlib import Path
import pandas as pd
import scanpy as sc
import commgwas

ROOT = Path('/gpfs/chencao/zhangcaicai/CCCGwas')
MAGMA_PATH = ROOT / 'magma/magma_cao/DisCP0423_1.magma.genes.results.csv'
H5AD_PATH = ROOT / 'data/Blood_Hao/qc.h5ad'
NICHENET_PATH = ROOT / 'resource/ligand_target_matrix_nsga2r_final.tsv'
CELL_TYPE_COL = 'predicted.celltype.l2'
MAGMA_GENE_COL = 'SYMBOL'
MAGMA_Z_COL = 'ZSTAT'
DATASET_ID = 'DisCP0423:1'

In [ ]:
magma = pd.read_csv(MAGMA_PATH)
adata = sc.read_h5ad(H5AD_PATH)
magma.head(), adata.shape, adata.obs[CELL_TYPE_COL].value_counts().head()

## Infer communication from AnnData

`infer_lr()` is the Scanpy-style entry point. It accepts the AnnData object directly and the cell-type column is explicit.

In [ ]:
lr_raw = commgwas.infer_lr(
    adata,
    cell_type_col=CELL_TYPE_COL,
    min_cells=20,
    min_expr_fraction=0.05,
    n_permutations=1000,
    random_state=0,
)
lr_raw.head()

## Select the LR set

The default is the requested CellPhoneDB rule `cellphone_pvals < 0.05`. No FDR is calculated here. If a LIANA or LIANA+ workflow needs another threshold, pass a pandas query instead.

In [ ]:
lr_sig = commgwas.filter_lr(lr_raw, threshold=0.05)
lr_sig.shape

In [ ]:
# Example of an alternative LIANA threshold; use only columns present in lr_raw.
lr_sig_strict = commgwas.filter_lr(
    lr_raw,
    filter_query='cellphone_pvals < 0.01 and aggregate_rank < 0.2',
)

## Tissue-level enrichment

All ligand and receptor genes in the selected LR table are tested together against the LR resource background.

In [ ]:
tissue_results = commgwas.enrich_tissue(
    lr_results=lr_sig,
    magma=magma,
    magma_gene_col=MAGMA_GENE_COL,
    magma_z_col=MAGMA_Z_COL,
    dataset=DATASET_ID,
    n_permutations=5000,
    random_state=0,
)
tissue_results

## Cell-type-pair enrichment

Each source-target pair is tested separately. The input table must contain LIANA-compatible `source` and `target` columns.

In [ ]:
cell_pair_results = commgwas.enrich_cell_type_pairs(
    lr_results=lr_sig,
    magma=magma,
    magma_gene_col=MAGMA_GENE_COL,
    magma_z_col=MAGMA_Z_COL,
    n_permutations=5000,
    random_state=0,
)
cell_pair_results.head()

## LR-pair enrichment

This level tests individual ligand-receptor pairs. The default MAGMA combination is the geometric mean of positive ligand and receptor z-scores.

In [ ]:
lr_results = commgwas.enrich_lr(
    lr_results=lr_sig,
    magma=magma,
    magma_gene_col=MAGMA_GENE_COL,
    magma_z_col=MAGMA_Z_COL,
    magma_mode='geom_mean',
    n_permutations=5000,
    random_state=0,
)
lr_results.head()

## NicheNet downstream enrichment

The downstream function accepts the AnnData object to enforce receiver-cell expression. It uses the NicheNet ligand-target matrix at `resource/ligand_target_matrix_nsga2r_final.tsv`. Passing `cellphone_p_cutoff=None` avoids refiltering because `lr_sig` was already selected above.

In [ ]:
downstream_results = commgwas.enrich_downstream(
    lr_results=lr_sig,
    adata=adata,
    magma=magma,
    nichenet=NICHENET_PATH,
    cell_type_col=CELL_TYPE_COL,
    magma_gene_col=MAGMA_GENE_COL,
    magma_z_col=MAGMA_Z_COL,
    cellphone_p_cutoff=None,
    top_n_targets=300,
    expression_fraction_cutoff=0.05,
    min_genes=10,
    n_permutations=5000,
    random_state=0,
)
downstream_results.head()

## Output contract

All result objects above are pandas DataFrames with raw `empirical_p` values. The package intentionally does not create `fdr`, `qvalue`, or Bonferroni columns; apply correction externally only if it is part of the analysis plan.